# SFSC 2026 — V5 Step 2: seed generator (YOLO11-seg, high resolution, 4 folds) (Colab / G4 or A100)

Step 2 of `v5/V5_PLAN.md`. The V5 refiner redraws every *seed* at native resolution, so seeds must find as many filaments as
possible; their outlines only need to be roughly right. Changes against V4's detector:

| | V4 detector | V5 seeds | Reason |
|---|---|---|---|
| model | YOLO11m-seg | **YOLO11l-seg** | more capacity for small and limb filaments |
| resolution | 1280 | **1536** | tiny filaments (49% missed) are 2-4 px wide at 1280 |
| input | raw JPEG | **3 channels: raw, limb-flattened, r/R** | V3's channels; the limb is where 53% of filaments are missed |
| score floor | 0.05 | **0.01**, up to 300 per image | recall first; the refiner's value head decides |

Same folds as V3 (`folds_v3.csv`), one training sample per annotator set, out-of-fold detections for every training image and
all-fold detections for every test image. The last cell reports **seed recall** (share of annotated filaments with a detection at
IoU > 0.3 / 0.5), by size and disk position, next to V4's detector. Plan gate: >= 90% at IoU > 0.3 (with V3 components added later).

**Run:** GPU runtime (G4 best, A100 fine). `Runtime -> Run all`. Resumable: after a disconnect, run all again; finished folds are
skipped and the running fold resumes from its last checkpoint on Drive. Outputs: `MyDrive/SFSC2026/outputs/sfsc_v5/seeds`.


## Cell 01 — Drive access, inputs (read only), output folder


In [ ]:
from pathlib import Path
import os, shutil, subprocess, threading

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_MODE = os.environ.get("SFSC_DRIVE_MODE", "mount" if IN_COLAB else "local")   # mount | rclone | local
RCLONE_REMOTE = os.environ.get("SFSC_RCLONE_REMOTE", "gdrive")
LOCAL_ROOT = Path(os.environ.get("SFSC_LOCAL_ROOT", "/content/sfsc_v5_seeds" if IN_COLAB else "/tmp/sfsc_v5_seeds"))
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA = LOCAL_ROOT / "dataset"
DEBUG = os.environ.get("SFSC_DEBUG", "0") == "1"
OUT_NAME = "seeds_debug" if DEBUG else "seeds"

def rclone(*args, check=True):
    cmd = ["rclone", *args, "--transfers", "16", "--checkers", "16", "--drive-chunk-size", "64M"]
    return subprocess.run(cmd, check=check)

if DRIVE_MODE == "mount":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SFSC2026")
elif DRIVE_MODE == "rclone":
    if shutil.which("rclone") is None:
        subprocess.run("curl -fsSL https://rclone.org/install.sh | bash", shell=True, check=True)
    DRIVE_ROOT = LOCAL_ROOT / "drive_mirror"
    remote = f"{RCLONE_REMOTE}:SFSC2026"
    rclone("copy", f"{remote}/data", str(DRIVE_ROOT / "data"), "--include", "/sfsc2026-data.zip")
    rclone("copy", f"{remote}/outputs/sfsc_unet_v3", str(DRIVE_ROOT / "outputs" / "sfsc_unet_v3"),
           "--include", "/folds_v3.csv", "--include", "/fold0/config.json")
    rclone("copy", f"{remote}/outputs/sfsc_fusion_v4", str(DRIVE_ROOT / "outputs" / "sfsc_fusion_v4"),
           "--include", "/detections/oof_*.pkl", "--include", "/detector/fold*/done.json")
    rclone("copy", f"{remote}/outputs/sfsc_v5/{OUT_NAME}", str(DRIVE_ROOT / "outputs" / "sfsc_v5" / OUT_NAME),
           "--exclude", "/detector/fold*/last.pt", check=False)               # resume state (finished folds, detections)
    rclone("copy", f"{remote}/outputs/sfsc_v5/{OUT_NAME}/detector", str(DRIVE_ROOT / "outputs" / "sfsc_v5" / OUT_NAME / "detector"),
           "--include", "/fold*/last.pt", check=False)                       # resume checkpoint of an unfinished fold
else:
    DRIVE_ROOT = Path(os.environ["SFSC_DRIVE_ROOT"])

DATA_ZIP = DRIVE_ROOT / "data" / "sfsc2026-data.zip"
DRIVE_EXTRACTED = DRIVE_ROOT / "data" / "SFSCDataset"
V3_RUN = DRIVE_ROOT / "outputs" / "sfsc_unet_v3"
V4_RUN = DRIVE_ROOT / "outputs" / "sfsc_fusion_v4"
RUN = DRIVE_ROOT / "outputs" / "sfsc_v5" / OUT_NAME
RUN.mkdir(parents=True, exist_ok=True)

_push_lock = threading.Lock()
def push(sub="", background=False):
    """rclone mode: copy RUN (or a sub-folder) back to Drive. Mount mode writes to Drive directly."""
    if DRIVE_MODE != "rclone":
        return
    def _go():
        with _push_lock:
            rclone("copy", str(RUN / sub), f"{RCLONE_REMOTE}:SFSC2026/outputs/sfsc_v5/{OUT_NAME}/{sub}".rstrip("/"), check=False)
    threading.Thread(target=_go, daemon=True).start() if background else _go()

print("Drive mode:", DRIVE_MODE, "| output:", RUN, "| debug:", DEBUG)


## Cell 02 — Dependencies


In [ ]:
import importlib.util, sys
REQUIRED = {"ultralytics": "ultralytics>=8.3.0", "pycocotools": "pycocotools>=2.0.7", "cv2": "opencv-python-headless>=4.8",
            "scipy": "scipy>=1.10", "pandas": "pandas>=2.0", "tqdm": "tqdm>=4.66"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
import torch, ultralytics
print("torch", torch.__version__, "| ultralytics", ultralytics.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## Cell 03 — Imports, configuration (edit here), helpers


In [ ]:
import copy, gc, hashlib, json, math, multiprocessing as mp, pickle, platform
import random, re, shutil, subprocess, sys, tempfile, time, warnings, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from dataclasses import dataclass, asdict
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mask_api
from tqdm.auto import tqdm


@dataclass
class Config:
    debug_run: bool = DEBUG                  # 2 epochs, fold 0 only, small inference subset
    local_dataset_override: str | None = os.environ.get("SFSC_DATASET") or None
    det_weights: str = "yolo11l-seg.pt"
    det_imgsz: int = int(os.environ.get("SFSC_IMGSZ", "1536"))
    det_epochs: int = 60
    det_batch: int = int(os.environ.get("SFSC_BATCH", "16"))
    det_cls_gain: float = 1.5
    det_workers: int = int(os.environ.get("SFSC_WORKERS", "24"))
    det_save_every: int = 2                  # checkpoint to Drive every N epochs
    det_folds: tuple = tuple(int(f) for f in os.environ.get("SFSC_FOLDS", "0,1,2,3").split(","))
    det_conf_floor: float = 0.01
    det_nms_iou: float = 0.6
    det_max_det: int = 300
    png_compression: int = 1

CFG = Config()
if CFG.debug_run:
    CFG.det_folds, CFG.det_epochs = (0,), 2
V3CFG = json.loads((V3_RUN / "fold0" / "config.json").read_text())["config"]
for _k in ("seed", "n_folds", "disk_radius_fraction", "radial_bins", "flat_median_kernel"):
    setattr(CFG, _k, V3CFG[_k])
CFG.expected_shape = tuple(int(x) for x in V3CFG["expected_shape"])
random.seed(CFG.seed); np.random.seed(CFG.seed)
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 2)
CFG.det_workers = min(CFG.det_workers, max(2, CPU_COUNT - 2))
T0 = time.time()

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1 << 20), b""):
            h.update(part)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".json_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(obj, f, indent=2, default=str)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def save_npy_atomic(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".npy_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "wb") as f:
            np.save(f, value, allow_pickle=False)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def copy_atomic(src, dst):
    """Copy unless a file of the same size is already there."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size == src.stat().st_size:
        return
    copy_force(src, dst)

def copy_force(src, dst):
    """Always copy (checkpoints of the same size differ in content)."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_suffix(dst.suffix + ".part")
    shutil.copyfile(src, temp)
    os.replace(temp, dst)

def save_pickle(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".part")
    with open(temp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp, path)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

print("CPU cores:", CPU_COUNT, "| config:", {k: v for k, v in asdict(CFG).items() if k.startswith("det_")})


## Cell 04 — Stage the dataset on the local SSD (unchanged from V4)


In [ ]:
ANNOTATION_BASENAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
LOCAL_ZIP = LOCAL_ROOT / "sfsc_dataset.zip"

def locate_dataset(root):
    root = Path(root)
    hits = sorted(root.rglob(ANNOTATION_BASENAME)) if root.is_dir() else []
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {ANNOTATION_BASENAME} under {root}; found {len(hits)}")
    ann = hits[0]
    train_dir = ann.parent / "train_images"
    test_dir = ann.parent.parent / "test" / "test_images"
    if not (train_dir.is_dir() and test_dir.is_dir()):
        raise FileNotFoundError(f"Missing train_images or test_images near {ann}")
    return ann, train_dir, test_dir

def image_paths(folder):
    return {p.name: p for p in Path(folder).iterdir()
            if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg")}

def staged_and_ready(root):
    try:
        a, t, s = locate_dataset(root)
        return a.stat().st_size > 1000 and len(image_paths(t)) > 0 and len(image_paths(s)) > 0
    except (FileNotFoundError, OSError):
        return False

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    resolved_root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in tqdm(archive.infolist(), desc="Extract dataset"):
            if member.is_dir():
                continue
            dest = (destination / member.filename).resolve()
            if not dest.is_relative_to(resolved_root):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
            if dest.is_file() and dest.stat().st_size == member.file_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            archive.extract(member, destination)

if CFG.local_dataset_override:
    DATA_ROOT = Path(CFG.local_dataset_override)
    if not staged_and_ready(DATA_ROOT):
        raise FileNotFoundError(f"Dataset override not ready: {DATA_ROOT}")
elif staged_and_ready(LOCAL_DATA):
    DATA_ROOT = LOCAL_DATA
else:
    DATA_ROOT = LOCAL_DATA
    if DATA_ZIP.is_file():
        print("Copying ZIP to local SSD once per runtime...")
        copy_atomic(DATA_ZIP, LOCAL_ZIP)
        safe_extract(LOCAL_ZIP, DATA_ROOT)
    elif DRIVE_EXTRACTED.is_dir():
        print("Copying extracted dataset folder from Drive to local SSD...")
        shutil.copytree(DRIVE_EXTRACTED, DATA_ROOT, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f"Expected {DATA_ZIP} or {DRIVE_EXTRACTED}. Or set CFG.local_dataset_override.")
if not staged_and_ready(DATA_ROOT):
    raise RuntimeError("Local staging incomplete. Check ZIP contents and free disk space.")

ANN_PATH, TRAIN_DIR, TEST_DIR = locate_dataset(DATA_ROOT)
TRAIN_PATHS, TEST_PATHS = image_paths(TRAIN_DIR), image_paths(TEST_DIR)
for label, paths in (("train", TRAIN_PATHS), ("test", TEST_PATHS)):
    if len({Path(n).stem for n in paths}) != len(paths):
        raise ValueError(f"Duplicate {label} image stems; IDs would collide.")

def read_gray(path):
    with Image.open(path) as im:
        if im.mode == "L":
            return np.asarray(im, dtype=np.uint8).copy()
        if im.mode != "RGB":
            raise ValueError(f"Unexpected image mode {im.mode}: {path}")
        arr = np.asarray(im, dtype=np.uint8)
        d1 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 1])
        d2 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 2])
        if d1.max() > 2 or d2.max() > 2:
            raise ValueError(f"Non-grayscale RGB image: {path}")
        return arr[:, :, 0].copy()

RAW = json.loads(ANN_PATH.read_text())
ANN_SHA = hash_file(ANN_PATH)
print("Annotation JSON:", ANN_PATH, "| sha256", ANN_SHA[:16])
print("JPEGs:", len(TRAIN_PATHS), "train /", len(TEST_PATHS), "test | free local GiB:",
      round(shutil.disk_usage(LOCAL_ROOT).free / 2**30, 1))


## Cell 05 — COCO parsing, record index (unchanged from V4)


In [ ]:
H, W = CFG.expected_shape

def native_rle(rle):
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    return {"size": [int(x) for x in rle["size"]], "counts": counts}

def json_rle(rle):
    counts = rle["counts"]
    return {"size": [int(x) for x in rle["size"]],
            "counts": counts.decode("ascii") if isinstance(counts, bytes) else counts}

def decode_mask(rle):
    a = mask_api.decode(native_rle(rle))
    return np.any(a, axis=2) if a.ndim == 3 else (a > 0)

def segmentation_to_rle(seg, h, w):
    if isinstance(seg, dict):
        if list(seg.get("size", [h, w])) != [h, w]:
            raise ValueError("Annotation RLE shape mismatch")
        rle = (mask_api.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list)
               else native_rle({"size": [h, w], "counts": seg["counts"]}))
    elif isinstance(seg, list) and len(seg):
        polys = seg if isinstance(seg[0], (list, tuple)) else [seg]
        if any(len(p) < 6 or len(p) % 2 for p in polys):
            raise ValueError("Malformed polygon segmentation")
        rle = mask_api.merge(mask_api.frPyObjects(polys, h, w))
    else:
        raise ValueError("Unsupported empty/malformed segmentation")
    return json_rle(rle)

STEMS = defaultdict(list)
for _n in TRAIN_PATHS:
    STEMS[Path(_n).stem].append(_n)

def canonical_filename(s):
    name = Path(str(s)).name
    if name in TRAIN_PATHS:
        return name
    cands = STEMS.get(Path(name).stem, [])
    if len(cands) == 1:
        return cands[0]
    raise ValueError(f"Unresolvable annotated JPEG: {s}")

IMAGES_BY_ID = {}
for im in RAW["images"]:
    rid = str(im["id"])
    if rid in IMAGES_BY_ID:
        raise ValueError(f"Duplicate image record {rid}")
    if (int(im["height"]), int(im["width"])) != (H, W):
        raise ValueError(f"Unexpected dimensions for record {rid}")
    IMAGES_BY_ID[rid] = im

ANNOTATIONS_BY_ID = defaultdict(list)
_seen = set()
for a in RAW["annotations"]:
    aid, rid = str(a["id"]), str(a["image_id"])
    if aid in _seen or rid not in IMAGES_BY_ID:
        raise ValueError(f"Duplicate annotation or missing image: {aid} / {rid}")
    if int(a.get("iscrowd", 0)):
        raise ValueError(f"Unexpected crowd annotation {aid}")
    if int(a.get("category_id", 1)) not in (1, 2, 3, 4):
        raise ValueError(f"Unexpected category {a.get('category_id')} in {aid}")
    _seen.add(aid)
    ANNOTATIONS_BY_ID[rid].append(a)

RECORD_IDS_BY_FILE = defaultdict(list)
for rid, im in IMAGES_BY_ID.items():
    RECORD_IDS_BY_FILE[canonical_filename(im["file_name"])].append(rid)
for k in RECORD_IDS_BY_FILE:
    RECORD_IDS_BY_FILE[k].sort()
UNIQUE_FILES = sorted(RECORD_IDS_BY_FILE)

FILE_HASHES = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    for name, digest in zip(UNIQUE_FILES, pool.map(lambda n: hash_file(TRAIN_PATHS[n]), UNIQUE_FILES)):
        FILE_HASHES[name] = digest

unannotated = sorted(set(TRAIN_PATHS) - set(UNIQUE_FILES))
if unannotated:
    warnings.warn(f"{len(unannotated)} train JPEGs have no annotation record; they are not used.")
print("Annotated JPEGs:", len(UNIQUE_FILES), "| annotator records:", len(IMAGES_BY_ID),
      "| instances:", len(RAW["annotations"]))
print("Records per JPEG:", pd.Series([len(v) for v in RECORD_IDS_BY_FILE.values()]).value_counts().sort_index().to_dict())


## Cell 06 — 3-channel detector images: raw, limb-flattened (V3's flattening), radial position r/R
Written once as PNG on the local SSD. The same file is read for training and for inference, so channel order is consistent.


In [ ]:
CY, CX = (H - 1) / 2.0, (W - 1) / 2.0
DISK_R = CFG.disk_radius_fraction * min(H, W) / 2.0
_YY, _XX = np.mgrid[0:H, 0:W]
RADIAL = (np.hypot(_YY - CY, _XX - CX) / DISK_R).astype(np.float32)
YY_FLAT = _YY.ravel().astype(np.float64)
XX_FLAT = _XX.ravel().astype(np.float64)
del _YY, _XX
DISK = RADIAL < 1.0
RBIN = np.minimum((RADIAL * CFG.radial_bins).astype(np.int32), CFG.radial_bins - 1)
RBIN_DISK = RBIN[DISK]

IMG_MEAN, IMG_STD = 0.5, 0.25
RAD_MEAN, RAD_STD, RAD_CLIP = 0.6, 0.35, 1.25

def flatten_disk(img):
    """Divide by the radial limb-darkening profile, then by a large local median; off-disk = neutral (1.0 -> 128)."""
    x = img.astype(np.float32)
    nb = CFG.radial_bins
    s = np.bincount(RBIN_DISK, weights=x[DISK], minlength=nb)
    c = np.bincount(RBIN_DISK, minlength=nb)
    valid = c > 0
    prof = np.interp(np.arange(nb), np.flatnonzero(valid), (s[valid] / c[valid]))
    k = 7
    prof = np.convolve(np.pad(prof, k // 2, mode="edge"), np.ones(k) / k, mode="valid")
    ratio = x / np.maximum(prof[RBIN], 1.0).astype(np.float32)
    ratio[~DISK] = 1.0
    small = cv2.resize(ratio, (W // 4, H // 4), interpolation=cv2.INTER_AREA)
    small_u8 = np.clip(small * 128.0, 0, 255).astype(np.uint8)
    ksize = max(3, int(round(CFG.flat_median_kernel * H / 2048)) | 1)
    med = cv2.medianBlur(small_u8, ksize).astype(np.float32) / 128.0
    med = cv2.resize(med, (W, H), interpolation=cv2.INTER_LINEAR)
    flat = ratio / np.maximum(med, 0.25)
    flat[~DISK] = 1.0
    return np.clip(flat * 128.0, 0, 255).astype(np.uint8)


FOLDS = pd.read_csv(V3_RUN / "folds_v3.csv", dtype={"file_name": str, "group": str, "v1_split": str})
if set(FOLDS.file_name) != set(UNIQUE_FILES):
    raise ValueError("V3's folds_v3.csv does not match this dataset.")
FOLD_OF = dict(zip(FOLDS.file_name, FOLDS.fold.astype(int)))
R_U8 = np.clip(RADIAL / RAD_CLIP * 255.0, 0, 255).astype(np.uint8)
PNG_DIR = LOCAL_ROOT / "png3"
PNG_DIR.mkdir(parents=True, exist_ok=True)

def write_png3(task):
    split, name, path = task
    out = PNG_DIR / f"{split}__{Path(name).stem}.png"
    if not out.is_file():
        img = read_gray(path)
        stack = np.dstack([img, flatten_disk(img), R_U8])
        tmp = out.with_suffix(".tmp.png")
        cv2.imwrite(str(tmp), stack, [cv2.IMWRITE_PNG_COMPRESSION, CFG.png_compression])
        os.replace(tmp, out)
    return name, str(out)

with ThreadPoolExecutor(max_workers=max(2, CPU_COUNT)) as pool:
    PNG_TRAIN = dict(tqdm(pool.map(write_png3, [("train", n, TRAIN_PATHS[n]) for n in UNIQUE_FILES]),
                          total=len(UNIQUE_FILES), desc="PNG train"))
    PNG_TEST = dict(tqdm(pool.map(write_png3, [("test", n, TEST_PATHS[n]) for n in sorted(TEST_PATHS)]),
                         total=len(TEST_PATHS), desc="PNG test"))
print("3-channel images:", len(PNG_TRAIN), "train /", len(PNG_TEST), "test")


## Cell 07 — Detector labels: one sample per annotator set, same 4 folds as V3 (V4 code, 3-channel images)


In [ ]:
YOLO_ROOT = LOCAL_ROOT / "yolo_v5"
(YOLO_ROOT / "images").mkdir(parents=True, exist_ok=True)
(YOLO_ROOT / "labels").mkdir(parents=True, exist_ok=True)

def outline_polygon(seg):
    """One outline per filament, in pixels: the annotated polygon (largest part if there are several)."""
    if isinstance(seg, list) and len(seg):
        polys = seg if isinstance(seg[0], (list, tuple)) else [seg]
        polys = [np.asarray(p, np.float64).reshape(-1, 2) for p in polys if len(p) >= 6]
        if polys:
            return max(polys, key=lambda p: abs(cv2.contourArea(p.astype(np.float32))))
    m = decode_mask(segmentation_to_rle(seg, H, W)).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = [c.reshape(-1, 2) for c in contours if len(c) >= 3]
    return max(contours, key=cv2.contourArea).astype(np.float64) if contours else None

def record_key(rid):
    return re.sub(r"[^A-Za-z0-9_-]", "_", str(rid))

# One training sample per annotator set: the same JPEG is linked once per set, each with that annotator's outlines.
RECORD_IMAGE, _n_obj = {}, 0
for name in tqdm(UNIQUE_FILES, desc="Detector labels"):
    src = Path(PNG_TRAIN[name]).resolve()
    for rid in RECORD_IDS_BY_FILE[name]:
        key = record_key(rid)
        link = YOLO_ROOT / "images" / f"{key}{src.suffix}"
        if not link.exists():
            try:
                os.symlink(src, link)
            except OSError:
                shutil.copyfile(src, link)
        lines = []
        for ann in ANNOTATIONS_BY_ID[rid]:
            poly = outline_polygon(ann["segmentation"])
            if poly is None or len(poly) < 3:
                continue
            xy = np.clip(poly / np.array([W, H], np.float64), 0.0, 1.0)
            lines.append("0 " + " ".join(f"{v:.6f}" for v in xy.ravel()))
        (YOLO_ROOT / "labels" / f"{key}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        RECORD_IMAGE[rid] = (name, str(link))
        _n_obj += len(lines)
if len({Path(p).name for _, p in RECORD_IMAGE.values()}) != len(RECORD_IMAGE):
    raise ValueError("Two annotator sets map to the same detector image name.")

DET_YAML = {}
for fold in range(CFG.n_folds):
    lists = {"train": [], "val": []}
    for rid, (name, link) in RECORD_IMAGE.items():
        lists["val" if FOLD_OF[name] == fold else "train"].append(link)
    for part, items in lists.items():
        (YOLO_ROOT / f"fold{fold}_{part}.txt").write_text("\n".join(sorted(items)) + "\n")
    DET_YAML[fold] = YOLO_ROOT / f"fold{fold}.yaml"
    DET_YAML[fold].write_text(f"path: {YOLO_ROOT}\ntrain: {YOLO_ROOT / f'fold{fold}_train.txt'}\n"
                              f"val: {YOLO_ROOT / f'fold{fold}_val.txt'}\nnames:\n  0: filament\n")
print("Detector samples (one per annotator set):", len(RECORD_IMAGE), "| outlines:", _n_obj)
print("Fold 0: train", len(Path(YOLO_ROOT / 'fold0_train.txt').read_text().split()),
      "| val", len(Path(YOLO_ROOT / 'fold0_val.txt').read_text().split()))


## Cell 08 — Train one YOLO11l-seg per fold (resumable; checkpoints copied to Drive every 2 epochs)


In [ ]:
import torch
from ultralytics import YOLO
try:
    from ultralytics import settings as _ul_settings
    _ul_settings.update({"wandb": False, "tensorboard": False, "comet": False, "mlflow": False, "clearml": False})
except Exception:
    pass
if not torch.cuda.is_available():
    raise RuntimeError("Seed training needs a GPU. Runtime -> Change runtime type -> G4 or A100.")
print("GPU:", torch.cuda.get_device_name(0))

DET_DIR = RUN / "detector"
YOLO_RUNS = LOCAL_ROOT / "yolo_runs"

def drive_saver(drive_dir):
    """Ultralytics callback: copy the resume and best checkpoints to Drive every few epochs."""
    def _cb(trainer):
        done = trainer.epoch + 1
        if done % CFG.det_save_every and done != trainer.epochs:
            return
        for src in (trainer.last, trainer.best):
            try:
                if Path(src).is_file():
                    copy_force(src, drive_dir / Path(src).name)
                    push(f"detector/{drive_dir.name}", background=True)
            except OSError as exc:
                print(f"  (could not copy {Path(src).name} to Drive: {exc})")
    return _cb

def train_detector(fold):
    drive_dir = DET_DIR / f"fold{fold}"
    drive_dir.mkdir(parents=True, exist_ok=True)
    done_path = drive_dir / "done.json"
    if done_path.exists():
        info = json.loads(done_path.read_text())
        print(f"Detector fold {fold}: already trained - skipping")
        return info
    local_run = YOLO_RUNS / f"fold{fold}"
    weights = local_run / "weights"
    t0 = time.perf_counter()
    if (drive_dir / "last.pt").is_file():
        weights.mkdir(parents=True, exist_ok=True)
        for w in ("last.pt", "best.pt"):
            if (drive_dir / w).is_file():
                copy_force(drive_dir / w, weights / w)
        print(f"Detector fold {fold}: resuming from the checkpoint on Drive")
        model = YOLO(str(weights / "last.pt"))
        model.add_callback("on_model_save", drive_saver(drive_dir))
        try:
            model.train(resume=True)
        except AssertionError as exc:            # raised when the checkpoint is from a finished run
            if "nothing to resume" not in str(exc) and "finished" not in str(exc):
                raise
            print("  checkpoint was already complete")
    else:
        model = YOLO(CFG.det_weights)
        model.add_callback("on_model_save", drive_saver(drive_dir))
        model.train(data=str(DET_YAML[fold]), imgsz=CFG.det_imgsz, epochs=CFG.det_epochs, batch=CFG.det_batch,
                    cls=CFG.det_cls_gain, workers=CFG.det_workers, seed=CFG.seed + fold, patience=CFG.det_epochs,
                    project=str(YOLO_RUNS), name=f"fold{fold}", exist_ok=True, plots=False, verbose=False,
                    flipud=0.5, fliplr=0.5, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2)
    final = weights / "best.pt" if (weights / "best.pt").is_file() else weights / "last.pt"
    if not final.is_file():
        raise FileNotFoundError(f"Detector fold {fold}: no weights were produced in {weights}")
    copy_force(final, drive_dir / "best.pt")
    info = {"fold": fold, "weights_sha256": hash_file(drive_dir / "best.pt"), "epochs": CFG.det_epochs,
            "imgsz": CFG.det_imgsz, "cls_gain": CFG.det_cls_gain, "minutes": round((time.perf_counter() - t0) / 60, 1)}
    write_json(done_path, info)
    (drive_dir / "last.pt").unlink(missing_ok=True)       # the finished fold no longer needs its resume checkpoint
    push(f"detector/{drive_dir.name}")
    del model
    gc.collect(); torch.cuda.empty_cache()
    print(f"Detector fold {fold}: done in {info['minutes']} min (this session)")
    return info

DET_INFO = {}
for _fold in CFG.det_folds:
    DET_INFO[_fold] = train_detector(_fold)


## Cell 09 — Detections: each training image by the fold model that never saw it, every test image by all fold models
Saved in V4's format (`{"scores", "polys"}` per image), so V4/V5 fusion code reads them unchanged.


In [ ]:
DET_FOLDS = sorted(int(p.parent.name.replace("fold", "")) for p in (RUN / "detector").glob("fold*/done.json"))
if not DET_FOLDS:
    raise RuntimeError("No trained fold found. Run Cell 08.")
DET_CACHE = RUN / "detections"
DET_CACHE.mkdir(parents=True, exist_ok=True)

def detect(model, path):
    """One image -> detection scores and outlines (pixel coordinates of the original image)."""
    r = model.predict(source=str(path), imgsz=CFG.det_imgsz, conf=CFG.det_conf_floor, iou=CFG.det_nms_iou,
                      max_det=CFG.det_max_det, retina_masks=True, verbose=False)[0]
    if r.masks is None or len(r.boxes) == 0:
        return {"scores": np.zeros(0, np.float32), "polys": []}
    scores = r.boxes.conf.detach().cpu().numpy().astype(np.float32)
    polys = [np.asarray(p, np.float32).reshape(-1, 2) for p in r.masks.xy]
    keep = [i for i, p in enumerate(polys) if len(p) >= 3]
    return {"scores": scores[keep], "polys": [polys[i] for i in keep]}


DET_OOF, _test_by_fold = {}, {}
for fold in DET_FOLDS:
    sha = json.loads((RUN / "detector" / f"fold{fold}" / "done.json").read_text())["weights_sha256"][:8]
    oof_file, test_file = DET_CACHE / f"oof_fold{fold}_{sha}.pkl", DET_CACHE / f"test_fold{fold}_{sha}.pkl"
    oof_names = sorted(n for n in UNIQUE_FILES if FOLD_OF[n] == fold)
    test_names = sorted(TEST_PATHS)
    if CFG.debug_run:
        oof_names, test_names = oof_names[:24], test_names[:8]
    oof = load_pickle(oof_file) if oof_file.is_file() else None
    test = load_pickle(test_file) if test_file.is_file() else None
    if oof is None or set(oof) != set(oof_names) or test is None or set(test) != set(test_names):
        local = LOCAL_ROOT / "det_weights" / f"fold{fold}_{sha}.pt"
        copy_atomic(RUN / "detector" / f"fold{fold}" / "best.pt", local)
        model = YOLO(str(local))
        oof = {n: detect(model, PNG_TRAIN[n]) for n in tqdm(oof_names, desc=f"Fold {fold}: out-of-fold images")}
        test = {n: detect(model, PNG_TEST[n]) for n in tqdm(test_names, desc=f"Fold {fold}: test images")}
        save_pickle(oof_file, oof); save_pickle(test_file, test)
        del model
        gc.collect(); torch.cuda.empty_cache()
    DET_OOF.update(oof)
    _test_by_fold[fold] = test
push("detections")
print("Folds:", DET_FOLDS, "| out-of-fold images:", len(DET_OOF), "| mean detections per image:",
      round(float(np.mean([len(d["scores"]) for d in DET_OOF.values()])), 1))


## Cell 10 — Seed recall: share of annotated filaments with a detection at IoU > 0.3 / 0.5 (V5 seeds vs V4 detector)
Counted per annotator set, as the metric does. Detection masks are taken at the 0.01 score floor (every seed the refiner sees).


In [ ]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

def det_windows(dets, floor):
    out = []
    for s, p in zip(dets["scores"], dets["polys"]):
        if s < floor or p is None or len(p) < 3:
            continue
        q = np.rint(np.asarray(p, np.float64)).astype(np.int32)
        x0, y0 = max(int(q[:, 0].min()), 0), max(int(q[:, 1].min()), 0)
        x1, y1 = min(int(q[:, 0].max()) + 1, W), min(int(q[:, 1].max()) + 1, H)
        if x1 - x0 < 2 or y1 - y0 < 2:
            continue
        roi = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.fillPoly(roi, [q - np.array([x0, y0], np.int32)], 1)
        out.append((y0, x0, roi.astype(bool)))
    return out

def recall_task(task):
    name, det_sets = task
    rows = []
    wins = {k: det_windows(d, 0.0) for k, d in det_sets.items()}
    for rid in RECORD_IDS_BY_FILE[name]:
        for ann in ANNOTATIONS_BY_ID[rid]:
            m = decode_mask(segmentation_to_rle(ann["segmentation"], H, W))
            ys, xs = np.nonzero(m)
            a = float(len(ys))
            r = math.hypot(ys.mean() - CY, xs.mean() - CX) / DISK_R
            row = {"file_name": name, "area": a, "radial": r}
            for k, win in wins.items():
                best = 0.0
                for y0, x0, w in win:
                    inside = (ys >= y0) & (ys < y0 + w.shape[0]) & (xs >= x0) & (xs < x0 + w.shape[1])
                    if inside.any():
                        i = float(w[ys[inside] - y0, xs[inside] - x0].sum())
                        best = max(best, i / (a + float(w.sum()) - i))
                row[f"iou_{k}"] = best
            rows.append(row)
    return rows

V4_OOF = {}
for p in sorted((V4_RUN / "detections").glob("oof_fold*.pkl")):
    V4_OOF.update(load_pickle(p))
names = sorted(DET_OOF)
tasks = [(n, {"v5": DET_OOF[n], **({"v4": V4_OOF[n]} if n in V4_OOF else {})}) for n in names]
with ProcessPoolExecutor(max_workers=max(2, CPU_COUNT - 1), mp_context=mp.get_context("fork")) as pool:
    REC = pd.DataFrame([r for rows in tqdm(pool.map(recall_task, tasks, chunksize=4), total=len(tasks), desc="Seed recall")
                        for r in rows])
REC["size"] = pd.cut(REC.area, [0, 500, 1500, 5000, np.inf], labels=["tiny", "small", "medium", "large"], right=False)
REC["disk"] = pd.cut(REC.radial, [0, 0.6, 0.85, np.inf], labels=["centre", "mid", "limb"], right=False)
summary = {}
for k in [c[4:] for c in REC.columns if c.startswith("iou_")]:
    col = REC[f"iou_{k}"]
    summary[k] = {"recall@0.3": float((col > 0.3).mean()), "recall@0.5": float((col > 0.5).mean()),
                  "by_size@0.3": REC.groupby("size", observed=True)[f"iou_{k}"].apply(lambda c: float((c > 0.3).mean())).to_dict(),
                  "by_disk@0.3": REC.groupby("disk", observed=True)[f"iou_{k}"].apply(lambda c: float((c > 0.3).mean())).to_dict()}
    print(f"{k}: recall IoU>0.3 {summary[k]['recall@0.3']:.3f} | IoU>0.5 {summary[k]['recall@0.5']:.3f}"
          f" | by size {({s: round(v, 3) for s, v in summary[k]['by_size@0.3'].items()})}"
          f" | by disk {({s: round(v, 3) for s, v in summary[k]['by_disk@0.3'].items()})}")
REC.to_csv(RUN / "seed_recall_instances.csv", index=False)
write_json(RUN / "seed_recall.json", {"filaments": len(REC), "images": len(names), "summary": summary})
push()


## Cell 11 — Manifest


In [ ]:
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
(RUN / "environment_seeds.txt").write_text(freeze.stdout)
write_json(RUN / "manifest_seeds.json", {
    "algorithm": "V5 seeds: YOLO11l-seg, 3-channel (raw, flat, r/R), V3 folds, one sample per annotator set",
    "config": asdict(CFG), "folds": DET_FOLDS, "detector_info": DET_INFO,
    "gpu": torch.cuda.get_device_name(0), "seed_recall": summary, "minutes": (time.time() - T0) / 60})
push()
print("Artifacts:", RUN, "| minutes:", round((time.time() - T0) / 60, 1))
